# Komentarų agresijos filtras – treniravimas Google Colab

**Prieš pradedant:** *Runtime → Change runtime type → T4 GPU*.

Užrašai paleidžia visą eksperimentą: duomenų paruošimas → EDA → Baseline → DistilBERT → palyginimas → XAI.

In [ ]:
!nvidia-smi -L

## 1. Kodas
Pakeiskite nuorodą į savo grupės GitHub repozitoriją.

In [ ]:
!git clone https://github.com/JUSU-VARDAS/agresijos-filtras.git
%cd agresijos-filtras
!pip install -q lime tabulate

## 2. Duomenys
**A variantas:** įkelkite `kaggle.json` (Kaggle → Settings → API → Create New Token). Prieš tai konkurso puslapyje reikia paspausti *Join / accept rules*.

**B variantas:** įkelkite `train.csv` rankiniu būdu (failas ~68 MB) ir praleiskite kitą langelį.

In [ ]:
from google.colab import files
import os
up = files.upload()  # pasirinkite kaggle.json ARBA train.csv
if 'kaggle.json' in up:
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !kaggle competitions download -c jigsaw-toxic-comment-classification-challenge -f train.csv.zip -p data/raw
    !unzip -o -q data/raw/train.csv.zip -d data/raw
elif 'train.csv' in up:
    os.replace('train.csv', 'data/raw/train.csv')
!ls -lh data/raw

## 3. Paruošimas ir EDA

In [ ]:
!python -m src.data
!python -m src.eda

## 4. Bazinis eksperimentas (TF-IDF + LogisticRegression)
Trunka ~5–10 min (CPU).

In [ ]:
!python -m src.train_baseline

## 5. Patobulintas eksperimentas (DistilBERT)
50 000 komentarų × 2 epochos ≈ 15–25 min T4 GPU. Visam rinkiniui: `--max-train 0` (~1 val.).

In [ ]:
!python -m src.train_distilbert --max-train 50000 --epochs 2

## 6. Palyginimas ir paaiškinamumas

In [ ]:
!python -m src.evaluate
!python -m src.explain

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open('reports/metrics/comparison.md').read()))
for f in ['metrics_comparison','confusion_matrices','roc_pr_curves','categories_f1','distilbert_training','bias_test','eda_labels','eda_toxic_words','xai_baseline_global','xai_occlusion_0']:
    p = f'reports/figures/{f}.png'
    if os.path.exists(p):
        display(Markdown(f'### {f}')); display(Image(p, width=700))

## 7. Išsaugoti rezultatus
Atsisiųskite modelius ir ataskaitos grafikus (modelių į GitHub nekelkite – per dideli).

In [ ]:
!zip -q -r rezultatai.zip models reports
files.download('rezultatai.zip')